# Othello AlphaZero on Google Colab — Universal Game Engine

Universal Game Engine のバックエンド（Bun + gRPC）を **Colab 内で起動**し、
Python (PyTorch) の **AlphaZero 風エージェント**（Policy/Value ネット + MCTS）が自己対戦で学習します。
木探索のノード展開はすべて gRPC の `BatchSimulate` で行い、Python 側はオセロのルールを持ちません。
学習済みモデルは Google Drive に保存されます。

**手順**: ランタイム → 「ランタイムのタイプを変更」で GPU (T4) を選んでから、上から順に実行してください。
DQN 版は `othello_dqn_colab.ipynb` を参照。

## 1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/UniversalGameEngine/models'
import os; os.makedirs(MODEL_DIR, exist_ok=True)
print('models will be saved to', MODEL_DIR)

## 2. リポジトリの取得と Bun のインストール

private リポジトリの場合は `REPO_URL` を `https://<GITHUB_TOKEN>@github.com/...` の形式にしてください。

In [ ]:
import os
REPO_URL = 'https://github.com/takumi-mr/UniversalGameEngine.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

if not os.path.exists('/content/UniversalGameEngine'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/UniversalGameEngine
%cd /content/UniversalGameEngine

# Bun
!curl -fsSL https://bun.sh/install | bash > /dev/null 2>&1
os.environ['PATH'] = '/root/.bun/bin:' + os.environ['PATH']
!bun --version

# 依存関係（postinstall の git hook 設定は Colab では不要なのでスキップ）
!bun install --frozen-lockfile --ignore-scripts

## 3. バックエンドをバックグラウンド起動

`RL_MODE=true` にすると Redis / MongoDB なしのインメモリ動作になります。ログは `server.log` に出ます。

In [ ]:
import subprocess, sys, time
sys.path.insert(0, '/content/UniversalGameEngine/apps/ml')

env = dict(os.environ, RL_MODE='true', PORT='3000', GRPC_PORT='50051')
server = subprocess.Popen(
    ['bun', 'run', 'apps/backend/server.ts'],
    cwd='/content/UniversalGameEngine',
    env=env,
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

from uge_rl.env import wait_for_server
wait_for_server('localhost:50051', timeout_sec=90)
print('gRPC server ready (pid', server.pid, ')')
!tail -n 5 /content/server.log

## 4. Python 依存関係

In [ ]:
!pip install -q -r apps/ml/requirements.txt
import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

## 5. 学習

1 イテレーション = 自己対戦 `--games-per-iter` 局 → `--train-steps-per-iter` 回の勾配更新。
`--simulations` は 1 手あたりの探索回数（大きいほど強いが遅い）。T4 では 1 イテレーション（20 局・100 探索）がおよそ 1〜2 分です。
中断した場合は `--resume {MODEL_PATH}` で再開できます。

In [ ]:
ITERATIONS = 40  #@param {type:"integer"}
GAMES_PER_ITER = 20  #@param {type:"integer"}
SIMULATIONS = 100  #@param {type:"integer"}
MODEL_PATH = f'{MODEL_DIR}/othello_az.pt'

!cd apps/ml && python -m uge_rl.train_az     --game othello     --address localhost:50051     --iterations {ITERATIONS}     --games-per-iter {GAMES_PER_ITER}     --simulations {SIMULATIONS}     --sim-batch 32     --train-steps-per-iter 200     --eval-every 5 --eval-games 20 --eval-simulations 50     --out {MODEL_PATH}

## 6. 評価（ランダムプレイヤーとの対戦）

In [ ]:
!cd apps/ml && python -m uge_rl.evaluate --checkpoint {MODEL_PATH} --address localhost:50051 --games 50

## 7. 学習曲線（対ランダム勝率）

In [ ]:
import json
import matplotlib.pyplot as plt

meta = json.load(open(MODEL_PATH.replace('.pt', '.json')))
hist = meta.get('eval_history', [])
if hist:
    games, wr = zip(*hist)
    plt.plot(games, wr, marker='o')
    plt.axhline(0.5, ls='--', c='gray')
    plt.xlabel('self-play games'); plt.ylabel('win rate vs random'); plt.ylim(0, 1)
    plt.title(f"Othello AlphaZero ({meta['games_played']} games, {meta['train_steps']} train steps)")
    plt.show()
print({k: meta[k] for k in ('format', 'arch', 'games_played', 'train_steps', 'saved_at', 'git_commit')})

## 8. 保存されたファイル

- `othello_az.pt` — Policy/Value ネットの state_dict + メタ情報（`uge_rl.checkpoint.load_checkpoint` で復元。`format: uge-rl/az/v1`）
- `othello_az.json` — メタ情報のみ

In [ ]:
!ls -la {MODEL_DIR}

## 9. 後片付け（任意）

In [ ]:
server.terminate()
print('server stopped')